# 眼底血管分割 v9 复现

已提交版本的 Kaggle Public Dice 为 **0.86017**，固定五折验证的提交尺度 Dice 为 **0.851748**。新模型按原图划分五折，每折训练 3,000 步，推理使用 EMA 权重和 D4 八种变换。

挂载本 Notebook 的 `v9_kaggle_assets.zip` 输入后运行全部单元格。默认 `replay` 从保存的模型概率重放融合与题目编码，检查生成的 `/kaggle/working/submission.csv` 是否与已提交 v9 逐字节相同。

要从图像重新推理，另外挂载原始比赛数据，将第一段代码的 `MODE` 改为 `"infer"`。该模式使用五个双域模型检查点和 v3 冻结概率。不同设备上的浮点结果可能导致少量像素差异。

In [ ]:
import hashlib
import json
import os
import zipfile
from pathlib import Path

MODE = os.environ.get("V9_KAGGLE_MODE", "replay")  # replay 或 infer
INPUT_ROOT = Path(os.environ.get("V9_KAGGLE_INPUT", "/kaggle/input"))
WORK_ROOT = Path(os.environ.get("V9_KAGGLE_WORK", "/kaggle/working"))
PROJECT = WORK_ROOT / "v9_project"
ASSET_NAME = 'v9_kaggle_assets.zip'
MANIFEST_NAME = 'v9_kaggle_assets_manifest.json'
EXPECTED_MANIFEST = '609078cbe329254f79bf6ed6e80626d0a3053fed8bb0219f954af7f766803696'

def install_inputs(read_bytes):
    manifest_bytes = read_bytes(MANIFEST_NAME)
    if hashlib.sha256(manifest_bytes).hexdigest() != EXPECTED_MANIFEST:
        raise ValueError("v9 输入清单校验失败")
    for name, expected in json.loads(manifest_bytes).items():
        content = read_bytes(name)
        if hashlib.sha256(content).hexdigest() != expected:
            raise ValueError(f"v9 输入校验失败：{name}")
        target = PROJECT / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(content)

archives = sorted(INPUT_ROOT.rglob(ASSET_NAME))
manifests = sorted(INPUT_ROOT.rglob(MANIFEST_NAME)) if not archives else []
if len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as archive:
        install_inputs(archive.read)
elif len(manifests) == 1:
    directory = manifests[0].parent
    install_inputs(lambda name: (directory / name).read_bytes())
else:
    raise FileNotFoundError(f"请在 Kaggle Notebook 的输入中挂载一份 {ASSET_NAME}")
print("v9 输入及 SHA-256 校验通过")

## 双域高分辨率模型

输入为 RGB、视野内标准化绿色通道和绿色局部对比度。两个预测头分别对应细节标签和提交尺度标签，训练用 BCE 与软 Dice。全分辨率特征融合参考 [FRD-Net](https://pmc.ncbi.nlm.nih.gov/articles/PMC11161363/)。下方为项目中实际使用的网络和损失。

In [ ]:
"""High-resolution multiscale vessel network with native and submission heads.

Independently implemented for this project. Full-resolution fusion is inspired
by FRD-Net (doi:10.1364/BOE.522482); this is not a reproduction of that model.
"""

from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F


class ConvNorm(nn.Sequential):
    def __init__(self, inputs, outputs, kernel=3, dilation=1):
        super().__init__(nn.Conv2d(inputs, outputs, kernel, padding=dilation * (kernel // 2),
                                  dilation=dilation, bias=False),
                         nn.BatchNorm2d(outputs, momentum=.03), nn.SiLU(inplace=True))


class Residual(nn.Module):
    def __init__(self, inputs, outputs, drop=0.):
        super().__init__()
        self.body = nn.Sequential(ConvNorm(inputs, outputs), nn.Dropout2d(drop),
                                  nn.Conv2d(outputs, outputs, 3, padding=1, bias=False),
                                  nn.BatchNorm2d(outputs, momentum=.03))
        self.skip = nn.Conv2d(inputs, outputs, 1, bias=False) if inputs != outputs else nn.Identity()

    def forward(self, x):
        return F.silu(self.body(x) + self.skip(x))


class DualDomainNet(nn.Module):
    def __init__(self, channels=(24, 40, 64, 96), drop=.1):
        super().__init__()
        self.encoders = nn.ModuleList([Residual(i, o, drop if n else 0.)
                                      for n, (i, o) in enumerate(zip((5, *channels[:-1]), channels))])
        self.context = nn.ModuleList([ConvNorm(channels[-1], 32, dilation=d) for d in (1, 2, 3)])
        self.context_mix = ConvNorm(96, channels[-1], 1)
        self.decoders = nn.ModuleList([Residual(i + o, o, drop)
                                      for i, o in zip(reversed(channels[1:]), reversed(channels[:-1]))])
        self.projections = nn.ModuleList([ConvNorm(c, 8, 1) for c in reversed(channels)])
        self.fuse = Residual(32, 24)
        self.native_head = nn.Conv2d(24, 1, 1)
        self.expansion_head = nn.Conv2d(24, 1, 1)
        self.aux_head = nn.Conv2d(channels[1], 1, 1)
        nn.init.constant_(self.expansion_head.bias, -1.5)

    def forward(self, image, auxiliary=False):
        if image.shape[-2] % 8 or image.shape[-1] % 8:
            raise ValueError("DualDomainNet requires dimensions divisible by eight")
        features = []
        x = image
        for index, encoder in enumerate(self.encoders):
            x = encoder(x if index == 0 else F.avg_pool2d(x, 2))
            features.append(x)
        x = x + self.context_mix(torch.cat([branch(x) for branch in self.context], dim=1))
        decoded = [x]
        for stage, skip in zip(self.decoders, reversed(features[:-1])):
            x = stage(torch.cat([F.interpolate(x, size=skip.shape[-2:], mode="nearest"), skip], dim=1))
            decoded.append(x)
        fused = self.fuse(torch.cat([F.interpolate(project(feature), size=image.shape[-2:], mode="nearest")
                                    for project, feature in zip(self.projections, decoded)], dim=1))
        native = self.native_head(fused)
        # The provided resize/>0 transform expands the nearest-neighbor mask.
        # A nonnegative logit correction represents this ordered pair of targets.
        submitted = native + F.softplus(self.expansion_head(fused))
        result = torch.cat([native, submitted], dim=1)
        if auxiliary:
            return result, self.aux_head(decoded[-2])
        return result


def binary_loss(logits, target):
    logits, target = logits.float(), target.float()
    p = logits.sigmoid()
    intersection = (p * target).sum((2, 3))
    dice = ((2 * intersection + 1) / (p.sum((2, 3)) + target.sum((2, 3)) + 1)).mean()
    return .5 * F.binary_cross_entropy_with_logits(logits, target) + .5 * (1 - dice)


def dual_loss(outputs, targets):
    logits, auxiliary = outputs
    native = binary_loss(logits[:, :1], targets[:, :1])
    submission = binary_loss(logits[:, 1:2], targets[:, 1:2])
    # Area targets preserve the occupancy of thin vessels in the coarse head.
    coarse = F.interpolate(targets[:, :1], size=auxiliary.shape[-2:], mode="area")
    return .4 * native + .6 * submission + .1 * binary_loss(auxiliary, coarse)

## 提交尺度转换

对 v3 原尺寸概率取不同阈值的掩码，按题目脚本用 Pillow 双线性缩放到 512×512 后取非零前景。转换后的 0.5 阈值掩码与 v3 脚本兼容提交的掩码一致。

In [ ]:
import numpy as np
from PIL import Image


def script_mask(mask: np.ndarray, size=(512, 512)) -> np.ndarray:
    """Pillow uint8 bilinear resize, then >0, exactly as turn_to_str does."""
    pixels = np.asarray(mask, dtype=np.uint8) * 255
    return np.asarray(Image.fromarray(pixels).resize(size, Image.Resampling.BILINEAR)) > 0


def transport(probabilities, threshold=.54):
    threshold = float(threshold)
    values = []
    for source in probabilities:
        score = np.zeros((512, 512), dtype=np.float32)
        for t in np.round(np.arange(.02, 1., .02), 2):
            score[script_mask(source >= float(t))] = float(t)
        score = (score + .01).clip(.001, .999)
        # Shift only the logit intercept so .54 in v3 becomes .5 here.
        odds = score / (1 - score) * ((1 - threshold) / threshold)
        calibrated = odds / (1 + odds)
        exact = script_mask(source >= threshold)
        if not np.array_equal(calibrated >= .5, exact):
            raise ValueError("Transport must preserve the deployed baseline mask exactly")
        values.append(calibrated)
    return np.stack(values)

## 读取概率或重新推理

从样例提交读取 ID 顺序。重新推理时校验比赛图像和五个检查点，并按固定顺序平均双域模型的提交头概率。

In [ ]:
import csv
import sys

sys.path.insert(0, str(PROJECT))
from vessel.runtime import seed_everything
from vessel_lab.dual_domain_study import load_grid, predict
from vessel_lab.submission_domain import export_masks, sha256

torch.set_num_threads(4)
recipe = json.loads((PROJECT / 'submission_v9_calibrated/inference_recipe.json').read_text(encoding="utf-8"))
model_recipe = json.loads((PROJECT / 'submission_v8_dual_domain/inference_recipe.json').read_text(encoding="utf-8"))
with (PROJECT / "sample_submission.csv").open(encoding="utf-8-sig", newline="") as stream:
    ids = [row["Id"] for row in csv.DictReader(stream)]

def read_probabilities(path, expected_shape):
    with np.load(path, allow_pickle=False) as archive:
        stored_ids = archive["ids"].astype(str).tolist()
        probabilities = archive["probabilities"]
    if len(stored_ids) != len(ids) or set(stored_ids) != set(ids):
        raise ValueError(f"概率文件 ID 与样例提交不符：{path.name}")
    if probabilities.shape != (len(ids), *expected_shape) or not np.isfinite(probabilities).all():
        raise ValueError(f"概率文件尺寸或数值异常：{path.name}")
    return probabilities[[stored_ids.index(image_id) for image_id in ids]]

def original_data_root():
    manifest = json.loads((PROJECT / "experiments/dataset_manifest.json").read_text(encoding="utf-8"))
    records = [record for record in manifest["records"] if record["group"] == "test/image"]
    configured = os.environ.get("V9_KAGGLE_DATA")
    roots = [Path(configured)] if configured else sorted({
        image.parents[2] for image in INPUT_ROOT.rglob("1.jpg")
        if image.parts[-3:] == ("test", "image", "1.jpg")
    })
    for root in roots:
        if all((root / record["path"]).is_file()
               and sha256(root / record["path"]) == record["file_sha256"] for record in records):
            return root
    raise FileNotFoundError("请挂载与数据清单一致的原始比赛数据，其中应包含 test/image")

def run_five_folds(data_root):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    config = model_recipe["config"]
    seed_everything(config["seed"], True)
    images, _, _ = load_grid(data_root, ids, "test")
    total = None
    for entry in model_recipe["checkpoints"]:
        path = PROJECT / entry["path"]
        if sha256(path) != entry["sha256"]:
            raise ValueError(f"检查点 SHA-256 不匹配：{path.name}")
        with torch.serialization.safe_globals([torch.torch_version.TorchVersion]):
            checkpoint = torch.load(path, map_location="cpu", weights_only=True)
        if checkpoint["config"] != config or checkpoint["fold"]["fold"] != entry["fold"]:
            raise ValueError("检查点训练配置或折号不匹配")
        model = DualDomainNet(**config["model"]).to(device, memory_format=torch.channels_last)
        model.load_state_dict(checkpoint["model"])
        probabilities = predict(model, images, device, d4=True)
        total = probabilities if total is None else total + probabilities
        print(f"Fold {entry['fold']} 推理完成（{device.type}）", flush=True)
    return (total / len(model_recipe["checkpoints"]))[:, 1]

v3_native = read_probabilities(PROJECT / 'submission_v3/test_probabilities_native.npz', (584, 565))
if MODE == "replay":
    v8_submission = read_probabilities(PROJECT / 'submission_v8_dual_domain/frozen_submission_probabilities.npz', (512, 512))
elif MODE == "infer":
    v8_submission = run_five_folds(original_data_root())
else:
    raise ValueError("MODE 请选择 replay 或 infer")

## 融合、编码和结果验证

融合权重为 **65% v3 + 35% 双域模型**，最终阈值 **0.42**。将 512×512 二值预测按行展开，使用 1 基起点 RLE。导出时执行题目给出的 `turn_to_str`，检查每张掩码和解码结果一致。

In [ ]:
import shutil

v3_submission = transport(v3_native, threshold=.54)
candidate_weight = recipe["candidate_weight"]
fused = (1 - candidate_weight) * v3_submission + candidate_weight * v8_submission
masks = fused >= recipe["threshold"]
export_dir = WORK_ROOT / "v9_export"
checks = export_masks(PROJECT, export_dir, ids, masks)
exact = checks["csv_sha256"] == recipe["csv_sha256"]
if MODE == "replay" and not exact:
    raise ValueError("重放 CSV 的 SHA-256 与已提交 v9 不符")

output = WORK_ROOT / "submission.csv"
shutil.copyfile(export_dir / "submission.csv", output)
print(json.dumps({
    "file": str(output),
    "rows": checks["rows"],
    "supplied_encoder_match": checks["exact_supplied_encoder_match"],
    "sha256": checks["csv_sha256"],
    "matches_submitted_v9": exact,
}, ensure_ascii=False, indent=2))

## 训练代码和校准记录

输入包提供 `vessel_lab/dual_domain_study.py`、`vessel_lab/dual_domain_fusion.py` 和 `vessel_lab/fusion_calibration.py`，以及五折划分、参数和数据指纹。若需重训，另建工作副本并加入原始 `train` 和 `test` 目录，从工作副本根目录依次运行：

```bash
python -m vessel_lab.dual_domain_study
python -m vessel_lab.dual_domain_fusion
python -m vessel_lab.fusion_calibration
```

v3 的冻结概率来自原有 15 个检查点集成，配置见 `submission_v3/inference_recipe.json`。从 v3 权重重算这部分概率需原 v3 复现包。当前 Notebook 的重放路径以 v3 和双域模型的冻结概率为起点，推理路径以 v3 冻结概率和五个双域权重为起点。